In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder.appName("basic").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 12:32:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = [
    (1, "Alice", "Bangladesh", 25),
    (2, "Bob", "United States", 34),
    (3, "Charlie", "Bangladesh", 42),
]

In [4]:
df = spark.createDataFrame(data, ["customer_id", "name", "country", "age"])

In [5]:
df.show()

+-----------+-------+-------------+---+
|customer_id|   name|      country|age|
+-----------+-------+-------------+---+
|          1|  Alice|   Bangladesh| 25|
|          2|    Bob|United States| 34|
|          3|Charlie|   Bangladesh| 42|
+-----------+-------+-------------+---+



In [6]:
df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- age: long (nullable = true)



In [7]:
df.columns

['customer_id', 'name', 'country', 'age']

In [8]:
df.dtypes

[('customer_id', 'bigint'),
 ('name', 'string'),
 ('country', 'string'),
 ('age', 'bigint')]

In [10]:
df.select("name", (F.col("age") + 1).alias("next_year_age")).show()

+-------+-------------+
|   name|next_year_age|
+-------+-------------+
|  Alice|           26|
|    Bob|           35|
|Charlie|           43|
+-------+-------------+



In [11]:
df.filter((F.col("age") > 30) & (F.col("country") == "Bangladesh")).show()

+-----------+-------+----------+---+
|customer_id|   name|   country|age|
+-----------+-------+----------+---+
|          3|Charlie|Bangladesh| 42|
+-----------+-------+----------+---+



In [14]:
df.select("country").distinct().show()

+-------------+
|      country|
+-------------+
|   Bangladesh|
|United States|
+-------------+



In [17]:
df.groupBy("country").agg(
    F.count("*").alias("customer_count"),
    F.sum("age").alias("total_age"),
    F.max("age").alias("max_age"),
    F.min("age").alias("min_age"),
).show()

+-------------+--------------+---------+-------+-------+
|      country|customer_count|total_age|max_age|min_age|
+-------------+--------------+---------+-------+-------+
|   Bangladesh|             2|       67|     42|     25|
|United States|             1|       34|     34|     34|
+-------------+--------------+---------+-------+-------+



In [18]:
df.createOrReplaceTempView("customers")

In [21]:
spark.sql("""
    SELECT
      country,
      COUNT(*) AS customer_count,
      SUM(age) AS total_age
    FROM customers
    GROUP BY country
""").show()

+-------------+--------------+---------+
|      country|customer_count|total_age|
+-------------+--------------+---------+
|   Bangladesh|             2|       67|
|United States|             1|       34|
+-------------+--------------+---------+

